In [ ]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')

transactions = pd.read_parquet('/kaggle/input/notebooks/hahahaha34234/phase-1/transactions_clean.parquet')
articles = pd.read_parquet('/kaggle/input/notebooks/hahahaha34234/phase-2/articles_features.parquet')

transactions['t_dat'] = pd.to_datetime(transactions['t_dat'])

print(f"Transactions: {transactions.shape}")
print(f"Date range: {transactions['t_dat'].min()} to {transactions['t_dat'].max()}")

In [ ]:
print(articles.columns)
print(articles.head(3))
print('==='*60)
print(transactions.columns)
print(transactions.head(3))

In [ ]:
# Join transactions with articles to get product group
trans_art = transactions.merge(
    articles[['article_id', 'product_group_name']], 
    on='article_id', 
    how='left'
)

# Total revenue per category
category_revenue = (trans_art.groupby('product_group_name')['price']
                    .sum()
                    .sort_values(ascending=False))

print("Revenue by product category:")
print(category_revenue.round(2))

# Pick top 4 categories
top_categories = category_revenue.head(4).index.tolist()
print(f"\nTop 4 categories selected: {top_categories}")

In [ ]:
def build_weekly_series(category_name):
    cat_data = trans_art[trans_art['product_group_name'] == category_name].copy()
    
    # Aggregate weekly transaction count and revenue
    cat_data['week'] = cat_data['t_dat'].dt.to_period('W').dt.start_time
    
    weekly = (cat_data.groupby('week')
              .agg(transaction_count=('article_id', 'count'),
                   weekly_revenue=('price', 'sum'))
              .reset_index())
    
    weekly.columns = ['ds', 'transaction_count', 'weekly_revenue']
    weekly['ds'] = pd.to_datetime(weekly['ds'])
    
    return weekly

# Build and preview series for each top category
category_series = {}
for cat in top_categories:
    series = build_weekly_series(cat)
    category_series[cat] = series
    print(f"{cat}: {len(series)} weeks, avg weekly transactions: {series['transaction_count'].mean():.0f}")

In [ ]:
def fit_and_forecast(weekly_df, category_name, forecast_weeks=16):
    # Prophet expects columns named ds (date) and y (target)
    df_prophet = weekly_df[['ds', 'transaction_count']].copy()
    df_prophet.columns = ['ds', 'y']
    
    model = Prophet(
        seasonality_mode='multiplicative',  # fashion is multiplicative not additive
        yearly_seasonality=True,
        weekly_seasonality=False,           # weekly patterns not meaningful for fashion
        daily_seasonality=False,
        changepoint_prior_scale=0.1,        # flexibility of trend changes
        seasonality_prior_scale=10.0        # strength of seasonality
    )
    
    # Add a back to school seasonality - strong signal in fashion
    model.add_seasonality(
        name='back_to_school',
        period=365.25,
        fourier_order=3
    )
    
    model.fit(df_prophet)
    
    # Forecast forward
    future = model.make_future_dataframe(periods=forecast_weeks, freq='W')
    forecast = model.predict(future)
    
    return model, forecast, df_prophet

# Fit for all top categories
results = {}
for cat in top_categories:
    print(f"Fitting Prophet for: {cat}")
    model, forecast, df_prophet = fit_and_forecast(category_series[cat], cat)
    results[cat] = {
        'model': model,
        'forecast': forecast,
        'actual': df_prophet
    }
    print(f"  Done. Forecast covers: {forecast['ds'].min().date()} to {forecast['ds'].max().date()}")

In [ ]:
fig, axes = plt.subplots(len(top_categories), 1, figsize=(16, 5 * len(top_categories)))

colors = ['steelblue', 'coral', 'mediumseagreen', 'mediumpurple']

for i, cat in enumerate(top_categories):
    ax = axes[i]
    forecast = results[cat]['forecast']
    actual = results[cat]['actual']
    
    # Split into historical and future
    history_end = actual['ds'].max()
    hist_forecast = forecast[forecast['ds'] <= history_end]
    future_forecast = forecast[forecast['ds'] > history_end]
    
    # Plot actual
    ax.plot(actual['ds'], actual['y'], 
            color=colors[i], linewidth=1.5, label='Actual', alpha=0.8)
    
    # Plot historical fit
    ax.plot(hist_forecast['ds'], hist_forecast['yhat'], 
            color='black', linewidth=1, linestyle='--', label='Model Fit', alpha=0.6)
    
    # Plot future forecast
    ax.plot(future_forecast['ds'], future_forecast['yhat'], 
            color=colors[i], linewidth=2.5, linestyle='-', label='Forecast', alpha=1.0)
    
    # Confidence interval on future only
    ax.fill_between(future_forecast['ds'],
                    future_forecast['yhat_lower'],
                    future_forecast['yhat_upper'],
                    alpha=0.2, color=colors[i], label='95% Confidence Interval')
    
    # Mark the forecast start
    ax.axvline(x=history_end, color='red', linestyle=':', linewidth=1.5, label='Forecast Start')
    
    ax.set_title(f'{cat} — Weekly Transaction Forecast', fontsize=13, fontweight='bold')
    ax.set_xlabel('Date')
    ax.set_ylabel('Weekly Transactions')
    ax.legend(loc='upper left', fontsize=9)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.suptitle('H&M Demand Forecast by Product Category', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('/kaggle/working/demand_forecast.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
def get_peak_weeks(forecast, category_name, top_n=3):
    # Look only at the forecast period (future weeks)
    history_end = pd.to_datetime('2020-09-22')
    future = forecast[forecast['ds'] > history_end].copy()
    
    if future.empty:
        # If no future dates, use last 8 weeks of historical forecast
        future = forecast.tail(8).copy()
    
    # Top N peak weeks by predicted demand
    peaks = future.nlargest(top_n, 'yhat')[['ds', 'yhat', 'yhat_lower', 'yhat_upper']]
    peaks['category'] = category_name
    peaks['ds'] = peaks['ds'].dt.strftime('%Y-%m-%d')
    peaks['yhat'] = peaks['yhat'].round(0).astype(int)
    
    return peaks

print("PEAK DEMAND WEEKS BY CATEGORY")
print("=" * 60)

all_peaks = []
for cat in top_categories:
    peaks = get_peak_weeks(results[cat]['forecast'], cat)
    all_peaks.append(peaks)
    print(f"\n{cat}:")
    print(peaks[['ds', 'yhat']].to_string(index=False))

all_peaks_df = pd.concat(all_peaks, ignore_index=True)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, cat in enumerate(top_categories):
    ax = axes[i // 2][i % 2]
    model = results[cat]['model']
    forecast = results[cat]['forecast']
    
    # Extract yearly seasonality component
    yearly = forecast[['ds', 'yearly']].copy()
    yearly['month'] = yearly['ds'].dt.month
    monthly_avg = yearly.groupby('month')['yearly'].mean()
    
    month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                   'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
    
    bars = ax.bar(range(1, 13), monthly_avg.values, 
                  color=['#e74c3c' if v == monthly_avg.max() 
                         else '#3498db' for v in monthly_avg.values])
    
    ax.set_xticks(range(1, 13))
    ax.set_xticklabels(month_names, fontsize=9)
    ax.set_title(f'{cat}\nSeasonal Pattern', fontsize=11, fontweight='bold')
    ax.set_ylabel('Seasonal Effect (multiplicative)')
    ax.axhline(y=0, color='black', linewidth=0.8, linestyle='--')
    
    # Annotate peak month
    peak_month = monthly_avg.idxmax()
    ax.annotate(f'Peak: {month_names[peak_month-1]}',
                xy=(peak_month, monthly_avg.max()),
                xytext=(peak_month + 0.5, monthly_avg.max() * 0.9),
                fontsize=9, color='red', fontweight='bold')

plt.suptitle('Seasonal Patterns by Product Category', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/kaggle/working/seasonality_components.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("\nBUSINESS RECOMMENDATIONS FROM DEMAND FORECAST")
print("=" * 65)

recommendations = {
    'Garment Upper body': 'Pre-position inventory and launch email campaigns 2 weeks before peak. Highest volume category — any stockout directly impacts revenue.',
    'Garment Lower body': 'Coordinate promotions with Upper body peaks — customers frequently buy both in the same session.',
    'Garment Full body': 'Peak aligns with party and occasion season. Target Champions and Potential Loyalists with outfit bundles.',
    'Accessories': 'Counter-seasonal to main garments — use accessory promotions to maintain revenue during garment off-peak weeks.'
}

for cat in top_categories:
    peak_week = all_peaks_df[all_peaks_df['category'] == cat].iloc[0]['ds']
    pred_vol = all_peaks_df[all_peaks_df['category'] == cat].iloc[0]['yhat']
    rec = recommendations.get(cat, 'Run targeted campaign 2 weeks before peak demand week.')
    
    print(f"\nCategory : {cat}")
    print(f"Peak Week: {peak_week}")
    print(f"Predicted Volume: {pred_vol:,} transactions")
    print(f"Recommendation: {rec}")

In [ ]:
# Save forecast data for each category
all_forecasts = []
for cat in top_categories:
    fc = results[cat]['forecast'][['ds', 'yhat', 'yhat_lower', 'yhat_upper', 'trend', 'yearly']].copy()
    fc['category'] = cat
    all_forecasts.append(fc)

forecast_df = pd.concat(all_forecasts, ignore_index=True)
forecast_df.to_parquet('/kaggle/working/demand_forecasts.parquet', index=False)
all_peaks_df.to_parquet('/kaggle/working/peak_weeks.parquet', index=False)

print("Saved demand_forecasts.parquet")
print("Saved peak_weeks.parquet")
print(f"\nForecast dataframe shape: {forecast_df.shape}")
print(f"\nPeak weeks summary:")
print(all_peaks_df.to_string(index=False))